# Insurance Notebook: Loss Reserving with Chain-Ladder (Expanded)

This notebook provides a review-grade actuarial reserving workflow using a cumulative paid triangle.

What is included:
- realistic synthetic triangle generation,
- data quality diagnostics,
- deterministic chain-ladder projection,
- reserve adequacy diagnostics,
- bootstrap reserve distribution,
- risk margin and governance-oriented summaries.

## 0) Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(11)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1) Build synthetic cumulative paid triangle

In [ ]:
accident_years = np.arange(2014, 2026)  # 12 AYs
dev_periods = np.arange(1, 13)          # 12 dev periods

# AY-level ultimate severity with mild trend
base_ult = np.random.lognormal(mean=10.6, sigma=0.22, size=len(accident_years))
inflation_trend = np.linspace(0.92, 1.08, len(accident_years))
ultimates = base_ult * inflation_trend

# CDF pattern to ultimate
cdf_pattern = np.array([0.18, 0.33, 0.47, 0.60, 0.70, 0.78, 0.84, 0.89, 0.93, 0.96, 0.985, 1.00])

triangle = pd.DataFrame(index=accident_years, columns=dev_periods, dtype=float)
for i, ay in enumerate(accident_years):
    max_dev_observed = len(accident_years) - i
    for d in dev_periods[:max_dev_observed]:
        noise = np.random.normal(1.0, 0.025)
        cal_effect = 1.0 + 0.01 * np.sin((ay + d) / 3.5)
        triangle.loc[ay, d] = ultimates[i] * cdf_pattern[d - 1] * noise * cal_effect

# enforce cumulative monotonicity
triangle = triangle.cummax(axis=1)
triangle.round(0).head()

## 2) Triangle diagnostics

In [ ]:
# Monotonicity and missingness checks
violations = 0
for ay in triangle.index:
    row = triangle.loc[ay].dropna().to_numpy()
    if np.any(np.diff(row) < 0):
        violations += 1

missing_cells = int(triangle.isna().sum().sum())

diag = pd.Series(
    {
        "num_accident_years": len(triangle),
        "num_dev_periods": triangle.shape[1],
        "missing_cells": missing_cells,
        "monotonicity_violations": violations,
        "latest_diagonal_paid": sum(triangle.loc[ay, len(accident_years) - i] for i, ay in enumerate(accident_years)),
    }
)
diag

In [ ]:
plt.figure(figsize=(9, 5))
sns.heatmap(triangle, cmap="Blues", linewidths=0.2, linecolor="white")
plt.title("Cumulative Paid Triangle")
plt.xlabel("Development period")
plt.ylabel("Accident year")
plt.tight_layout()
plt.show()

## 3) Chain-ladder age-to-age factors

In [ ]:
link_factors = {}
for d in range(1, triangle.shape[1]):
    num = 0.0
    den = 0.0
    for ay in accident_years:
        if pd.notna(triangle.loc[ay, d]) and pd.notna(triangle.loc[ay, d + 1]):
            num += triangle.loc[ay, d + 1]
            den += triangle.loc[ay, d]
    link_factors[d] = num / den

f = pd.Series(link_factors, name="age_to_age")

# cumulative development factors to ultimate
cdf_to_ult = {triangle.shape[1]: 1.0}
prod = 1.0
for d in range(triangle.shape[1] - 1, 0, -1):
    prod *= f[d]
    cdf_to_ult[d] = prod

cdf = pd.Series(cdf_to_ult).sort_index()
cdf.name = "cdf_to_ultimate"

factor_table = pd.DataFrame(
    {
        "dev_period": np.arange(1, triangle.shape[1]),
        "age_to_age": [f[d] for d in range(1, triangle.shape[1])],
        "cdf_to_ultimate": [cdf[d] for d in range(1, triangle.shape[1])],
    }
)
factor_table

In [ ]:
plt.figure(figsize=(8, 4.2))
plt.plot(factor_table["dev_period"], factor_table["age_to_age"], marker="o", color="#4C72B0")
plt.title("Age-to-Age Factors")
plt.xlabel("Development period")
plt.ylabel("Link ratio")
plt.tight_layout()
plt.show()

## 4) Deterministic chain-ladder ultimates and IBNR

In [ ]:
rows = []
for i, ay in enumerate(accident_years):
    latest_dev = len(accident_years) - i
    latest_paid = triangle.loc[ay, latest_dev]
    selected_cdf = cdf.loc[latest_dev]

    ultimate = latest_paid * selected_cdf
    ibnr = ultimate - latest_paid

    rows.append(
        {
            "accident_year": ay,
            "latest_dev": latest_dev,
            "latest_paid": latest_paid,
            "selected_cdf": selected_cdf,
            "projected_ultimate": ultimate,
            "ibnr": ibnr,
        }
    )

cl = pd.DataFrame(rows)
cl

In [ ]:
total_latest_paid = cl["latest_paid"].sum()
total_ultimate = cl["projected_ultimate"].sum()
total_ibnr = cl["ibnr"].sum()

summary = pd.Series(
    {
        "total_latest_paid": total_latest_paid,
        "total_projected_ultimate": total_ultimate,
        "total_ibnr": total_ibnr,
        "ibnr_to_latest_paid_ratio": total_ibnr / total_latest_paid,
    }
)
summary

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.bar(cl["accident_year"].astype(str), cl["ibnr"], color="#55A868")
plt.title("IBNR by Accident Year")
plt.xlabel("Accident year")
plt.ylabel("IBNR")
plt.tight_layout()
plt.show()

## 5) Reserve concentration and maturity view

In [ ]:
cl["ibnr_share"] = cl["ibnr"] / cl["ibnr"].sum()
cl["maturity_bucket"] = pd.cut(
    cl["latest_dev"],
    bins=[0, 3, 6, 9, 12],
    labels=["early(1-3)", "mid(4-6)", "late(7-9)", "mature(10-12)"],
)

maturity = (
    cl.groupby("maturity_bucket", as_index=False, observed=False)
    .agg(total_ibnr=("ibnr", "sum"), ay_count=("accident_year", "size"))
)
maturity["share"] = maturity["total_ibnr"] / maturity["total_ibnr"].sum()

cl.sort_values("ibnr", ascending=False).head(6), maturity

## 6) Bootstrap uncertainty (link-ratio resampling)

In [ ]:
# Collect empirical link-ratio pools by development period
ratio_pool = {}
for d in range(1, triangle.shape[1]):
    ratio_pool[d] = np.array(
        [
            triangle.loc[ay, d + 1] / triangle.loc[ay, d]
            for ay in accident_years
            if pd.notna(triangle.loc[ay, d]) and pd.notna(triangle.loc[ay, d + 1])
        ]
    )

n_boot = 1500
boot_totals = []

for _ in range(n_boot):
    # bootstrap factors
    bf = {}
    for d in range(1, triangle.shape[1]):
        draws = np.random.choice(ratio_pool[d], size=max(len(ratio_pool[d]), 20), replace=True)
        bf[d] = np.mean(draws) * np.random.normal(1.0, 0.01)

    # convert to CDF
    bcdf = {triangle.shape[1]: 1.0}
    prod = 1.0
    for d in range(triangle.shape[1] - 1, 0, -1):
        prod *= bf[d]
        bcdf[d] = prod

    total_ibnr_boot = 0.0
    for i, ay in enumerate(accident_years):
        latest_dev = len(accident_years) - i
        latest_paid = triangle.loc[ay, latest_dev]
        total_ibnr_boot += latest_paid * bcdf[latest_dev] - latest_paid

    boot_totals.append(total_ibnr_boot)

boot_totals = np.array(boot_totals)

boot_stats = pd.Series(
    {
        "mean": float(np.mean(boot_totals)),
        "std": float(np.std(boot_totals, ddof=1)),
        "p50": float(np.quantile(boot_totals, 0.50)),
        "p75": float(np.quantile(boot_totals, 0.75)),
        "p90": float(np.quantile(boot_totals, 0.90)),
        "p95": float(np.quantile(boot_totals, 0.95)),
        "p99": float(np.quantile(boot_totals, 0.99)),
    }
)
boot_stats

In [ ]:
plt.figure(figsize=(8.5, 4.5))
plt.hist(boot_totals, bins=40, color="#8172B2", alpha=0.85)
plt.axvline(total_ibnr, color="black", linestyle="--", label="Deterministic CL")
plt.axvline(np.quantile(boot_totals, 0.95), color="#C44E52", linestyle="--", label="95th percentile")
plt.title("Bootstrap Distribution of Total IBNR")
plt.xlabel("Total IBNR")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

## 7) Risk margin and reserve adequacy lens

In [ ]:
risk_margin = np.quantile(boot_totals, 0.75) - total_ibnr
reserve_with_margin = total_ibnr + risk_margin

adequacy = pd.Series(
    {
        "deterministic_ibnr": total_ibnr,
        "risk_margin_p75_minus_mean": risk_margin,
        "reserve_including_margin": reserve_with_margin,
        "p95_minus_mean_stress_buffer": np.quantile(boot_totals, 0.95) - total_ibnr,
        "coefficient_of_variation": np.std(boot_totals, ddof=1) / np.mean(boot_totals),
    }
)
adequacy

## 8) Governance summary table

In [ ]:
governance = pd.DataFrame(
    {
        "metric": [
            "Deterministic IBNR",
            "Bootstrap mean IBNR",
            "Bootstrap std",
            "P75 IBNR",
            "P95 IBNR",
            "Largest AY IBNR share",
            "Top-3 AY IBNR share",
            "IBNR / Latest Paid",
        ],
        "value": [
            total_ibnr,
            np.mean(boot_totals),
            np.std(boot_totals, ddof=1),
            np.quantile(boot_totals, 0.75),
            np.quantile(boot_totals, 0.95),
            cl["ibnr_share"].max(),
            cl["ibnr_share"].sort_values(ascending=False).head(3).sum(),
            total_ibnr / total_latest_paid,
        ],
    }
)
governance

## 9) Final summary

- The deterministic chain-ladder estimate provides a central reserve view.
- Bootstrap uncertainty quantifies reserve volatility and downside buffers.
- Concentration and maturity diagnostics identify where reserve risk is most sensitive.
- The governance table can be used directly in reserve committee review packs.